<a href="https://colab.research.google.com/github/Addis-AI-Org/Notebooks/blob/main/hausa_swahili_asr_pilot_preview.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Addis AI Speech API Pilot Preview

This Colab notebook benchmarks live Addis AI ASR performance for **Hausa** and **Swahili** using a matched clean/noisy public evaluation set.

It reports:
- Word Error Rate (WER) and Character Error Rate (CER)
- End-to-end latency
- Real-time factor (RTF)
- Returned confidence scores
- 98% bootstrap confidence intervals

Methodology highlights:
- Clean evaluation uses `50` held-out public test utterances per language.
- Noisy evaluation reuses the same utterances with synthetic background interference at fixed `15 dB`, `10 dB`, and `5 dB` SNR.
- Headline metrics use normalized text: Unicode NFC, lowercase, whitespace collapse, and punctuation removal.
- This notebook is **ASR only** and is intended for customer-facing pilot review.

Notes:
- Noisy results reflect matched synthetic interference, not a naturally noisy field corpus.
- The notebook calls the public Addis AI Speech API directly; no backend changes are required.

In [ ]:
%pip -q install "datasets[audio]<4" huggingface_hub jiwer soundfile librosa scipy seaborn pandas numpy requests tqdm


In [ ]:
import getpass
import json
import math
import os
import random
import re
import time
import unicodedata
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
import soundfile as sf
from datasets import load_dataset
from huggingface_hub import HfApi, hf_hub_download
from IPython.display import Markdown, display
from jiwer import process_characters, process_words
from tqdm.auto import tqdm

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 6)
pd.set_option("display.max_colwidth", 220)

ADDIS_API_KEY = os.getenv("ADDIS_API_KEY") or getpass.getpass("Enter Addis AI API key: ").strip()
API_BASE_URL = os.getenv("API_BASE_URL", "https://api.addisassistant.com").rstrip("/")
SAMPLE_SIZE_PER_LANGUAGE = int(os.getenv("SAMPLE_SIZE_PER_LANGUAGE", "50"))
SEED = int(os.getenv("SEED", "42"))
MAX_CONCURRENCY = int(os.getenv("MAX_CONCURRENCY", "4"))
REQUEST_TIMEOUT_SECONDS = int(os.getenv("REQUEST_TIMEOUT_SECONDS", "120"))
MAX_RETRIES = int(os.getenv("MAX_RETRIES", "2"))
BOOTSTRAP_SAMPLES = int(os.getenv("BOOTSTRAP_SAMPLES", "2000"))

OUTPUT_DIR = Path("addis_ai_asr_pilot_preview")
CLEAN_AUDIO_DIR = OUTPUT_DIR / "audio_clean"
NOISY_AUDIO_DIR = OUTPUT_DIR / "audio_noisy"
NOISE_CACHE_DIR = OUTPUT_DIR / "noise_cache"

for path in [OUTPUT_DIR, CLEAN_AUDIO_DIR, NOISY_AUDIO_DIR, NOISE_CACHE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

LANGUAGE_CONFIGS = {
    "ha": {"label": "Hausa", "dataset_config": "ha_ng"},
    "sw": {"label": "Swahili", "dataset_config": "sw_ke"},
}
NOISE_REPO_IDS = [
    "FluidInference/musan",
    "alexwengg/musan_mini50",
]
NOISE_CATEGORY_ORDER = ["noise", "music", "babble"]
SNR_PLAN = [15] * 20 + [10] * 20 + [5] * 10

if not ADDIS_API_KEY:
    raise ValueError("ADDIS_API_KEY is required to run the benchmark.")
if SAMPLE_SIZE_PER_LANGUAGE != 50:
    display(Markdown(
        f"> Pilot note: `SAMPLE_SIZE_PER_LANGUAGE` is currently `{SAMPLE_SIZE_PER_LANGUAGE}` instead of the default `50`."
    ))

print(f"API base URL: {API_BASE_URL}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")


In [ ]:
def normalize_text(text: str, remove_punct: bool = False) -> str:
    text = unicodedata.normalize("NFC", text or "")
    text = re.sub(r"\s+", " ", text.strip().lower())
    if remove_punct:
        text = "".join(ch for ch in text if not unicodedata.category(ch).startswith("P"))
        text = re.sub(r"\s+", " ", text).strip()
    return text


def parse_confidence(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return np.nan


def bootstrap_ci(values, n_boot: int = BOOTSTRAP_SAMPLES, ci: float = 95.0, seed: int = SEED):
    arr = np.asarray([value for value in values if pd.notna(value)], dtype=float)
    if arr.size == 0:
        return (np.nan, np.nan)
    if arr.size == 1:
        return (float(arr[0]), float(arr[0]))
    rng = np.random.default_rng(seed)
    means = np.empty(n_boot, dtype=float)
    for idx in range(n_boot):
        sample = rng.choice(arr, size=arr.size, replace=True)
        means[idx] = float(np.mean(sample))
    alpha = (100 - ci) / 2
    return float(np.percentile(means, alpha)), float(np.percentile(means, 100 - alpha))


def summarize_metric(values, prefix: str):
    arr = np.asarray([value for value in values if pd.notna(value)], dtype=float)
    if arr.size == 0:
        return {
            f"{prefix}_mean": np.nan,
            f"{prefix}_median": np.nan,
            f"{prefix}_p95": np.nan,
            f"{prefix}_ci_low": np.nan,
            f"{prefix}_ci_high": np.nan,
        }
    ci_low, ci_high = bootstrap_ci(arr)
    return {
        f"{prefix}_mean": float(np.mean(arr)),
        f"{prefix}_median": float(np.median(arr)),
        f"{prefix}_p95": float(np.percentile(arr, 95)),
        f"{prefix}_ci_low": ci_low,
        f"{prefix}_ci_high": ci_high,
    }


def infer_noise_base_category(repo_path: str) -> str:
    normalized = repo_path.lower().strip("/")
    parts = normalized.split("/")
    if parts and parts[0] in {"music", "speech", "noise"}:
        return parts[0]
    for category in ["music", "speech", "noise"]:
        if f"/{category}/" in normalized:
            return category
    return "other"


def list_noise_inventory(repo_ids = NOISE_REPO_IDS) -> pd.DataFrame:
    api = HfApi()
    rows = []
    for repo_id in repo_ids:
        repo_files = api.list_repo_files(repo_id=repo_id, repo_type="dataset")
        for repo_path in repo_files:
            if not repo_path.lower().endswith(".wav"):
                continue
            base_category = infer_noise_base_category(repo_path)
            if base_category == "other":
                continue
            rows.append(
                {
                    "repo_id": repo_id,
                    "repo_path": repo_path,
                    "base_category": base_category,
                    "source_id": Path(repo_path).stem,
                }
            )
    inventory = pd.DataFrame(rows)
    if inventory.empty:
        raise RuntimeError("No compatible MUSAN noise clips were discovered.")
    return inventory


def ensure_noise_file(repo_id: str, repo_path: str) -> Path:
    local_path = hf_hub_download(
        repo_id=repo_id,
        repo_type="dataset",
        filename=repo_path,
        local_dir=NOISE_CACHE_DIR / repo_id.replace("/", "__"),
        local_dir_use_symlinks=False,
    )
    return Path(local_path)


def load_mono_audio(audio_path: Path, target_sr: int = 16000):
    audio, sampling_rate = sf.read(audio_path, dtype="float32")
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)
    if sampling_rate != target_sr:
        audio = librosa.resample(audio, orig_sr=sampling_rate, target_sr=target_sr)
        sampling_rate = target_sr
    peak = np.max(np.abs(audio)) if audio.size else 0.0
    if peak > 1.0:
        audio = audio / peak
    return audio.astype(np.float32), int(sampling_rate)


def fit_audio_length(audio: np.ndarray, target_length: int) -> np.ndarray:
    if audio.size == 0:
        return np.zeros(target_length, dtype=np.float32)
    if len(audio) >= target_length:
        return audio[:target_length]
    repeats = math.ceil(target_length / len(audio))
    return np.tile(audio, repeats)[:target_length].astype(np.float32)


def build_babble_signal(noise_refs, target_sr: int = 16000):
    layers = []
    for repo_id, repo_path in noise_refs:
        audio, _ = load_mono_audio(ensure_noise_file(repo_id, repo_path), target_sr=target_sr)
        layers.append(audio)
    target_length = max(len(layer) for layer in layers)
    combined = np.zeros(target_length, dtype=np.float32)
    for layer in layers:
        combined += fit_audio_length(layer, target_length)
    peak = np.max(np.abs(combined)) if combined.size else 0.0
    if peak > 0:
        combined = combined / peak
    return combined.astype(np.float32)


def mix_with_snr(clean_audio: np.ndarray, noise_audio: np.ndarray, snr_db: float) -> np.ndarray:
    if clean_audio.size == 0:
        return clean_audio.astype(np.float32)
    noise_audio = fit_audio_length(noise_audio, len(clean_audio))
    clean_rms = float(np.sqrt(np.mean(np.square(clean_audio))))
    noise_rms = float(np.sqrt(np.mean(np.square(noise_audio))))
    if clean_rms == 0.0 or noise_rms == 0.0:
        return clean_audio.astype(np.float32)
    desired_noise_rms = clean_rms / (10 ** (snr_db / 20.0))
    scaled_noise = noise_audio * (desired_noise_rms / noise_rms)
    mixed = clean_audio + scaled_noise
    peak = float(np.max(np.abs(mixed))) if mixed.size else 0.0
    if peak > 0.999:
        mixed = mixed / peak * 0.999
    return mixed.astype(np.float32)


def compute_error_metrics(reference: str, prediction: str):
    reference = reference or ""
    prediction = prediction or ""
    raw_word = process_words(reference, prediction)
    raw_char = process_characters(reference, prediction)
    normalized_reference = normalize_text(reference, remove_punct=True)
    normalized_prediction = normalize_text(prediction, remove_punct=True)
    normalized_word = process_words(normalized_reference, normalized_prediction)
    normalized_char = process_characters(normalized_reference, normalized_prediction)
    return {
        "reference_raw": reference,
        "prediction_raw": prediction,
        "reference_normalized": normalized_reference,
        "prediction_normalized": normalized_prediction,
        "wer_raw": float(raw_word.wer),
        "cer_raw": float(raw_char.cer),
        "wer_normalized": float(normalized_word.wer),
        "cer_normalized": float(normalized_char.cer),
        "word_substitutions": int(normalized_word.substitutions),
        "word_deletions": int(normalized_word.deletions),
        "word_insertions": int(normalized_word.insertions),
        "char_substitutions": int(normalized_char.substitutions),
        "char_deletions": int(normalized_char.deletions),
        "char_insertions": int(normalized_char.insertions),
    }


def extract_api_error(response_json) -> str:
    if isinstance(response_json, dict):
        if isinstance(response_json.get("error"), dict):
            message = response_json["error"].get("message")
            if message:
                return str(message)
        for key in ["message", "detail", "status", "raw_text"]:
            value = response_json.get(key)
            if value:
                return str(value)
    return "Request failed"


def call_stt_api(audio_path: Path, language_code: str):
    url = f"{API_BASE_URL}/api/v2/stt"
    payload = json.dumps({"language_code": language_code})
    last_error = None
    for attempt in range(1, MAX_RETRIES + 2):
        started = time.perf_counter()
        try:
            with audio_path.open("rb") as audio_file:
                response = requests.post(
                    url,
                    headers={"x-api-key": ADDIS_API_KEY},
                    files={"audio": (audio_path.name, audio_file, "audio/wav")},
                    data={"request_data": payload},
                    timeout=REQUEST_TIMEOUT_SECONDS,
                )
            latency_ms = (time.perf_counter() - started) * 1000
            try:
                response_json = response.json()
            except ValueError:
                response_json = {"raw_text": response.text[:2000]}

            if response.status_code in (401, 403):
                raise PermissionError("Authentication failed. Check the Addis AI API key and rerun the config cell.")

            if response.status_code >= 500 and attempt <= MAX_RETRIES:
                last_error = f"HTTP {response.status_code}: {extract_api_error(response_json)}"
                time.sleep(1.5 * attempt)
                continue

            data = response_json.get("data", {}) if isinstance(response_json, dict) else {}
            transcription = str(data.get("transcription") or "")
            usage_metadata = data.get("usage_metadata") if isinstance(data, dict) else None
            return {
                "http_status": int(response.status_code),
                "success": bool(response.ok and response_json.get("status") == "success") if isinstance(response_json, dict) else bool(response.ok),
                "latency_ms": float(latency_ms),
                "transcription": transcription,
                "confidence": parse_confidence(data.get("confidence") if isinstance(data, dict) else np.nan),
                "usage_metadata": usage_metadata,
                "error_message": None if response.ok else extract_api_error(response_json),
            }
        except PermissionError:
            raise
        except requests.Timeout:
            last_error = f"Timed out after {REQUEST_TIMEOUT_SECONDS} seconds"
        except Exception as exc:
            last_error = f"{type(exc).__name__}: {exc}"
        if attempt <= MAX_RETRIES:
            time.sleep(1.5 * attempt)
    return {
        "http_status": np.nan,
        "success": False,
        "latency_ms": np.nan,
        "transcription": "",
        "confidence": np.nan,
        "usage_metadata": None,
        "error_message": last_error or "Request failed",
    }


def prepare_clean_manifest(sample_size: int = SAMPLE_SIZE_PER_LANGUAGE, seed: int = SEED) -> pd.DataFrame:
    rng = random.Random(seed)
    manifest_rows = []
    for language_code, config in LANGUAGE_CONFIGS.items():
        dataset = load_dataset("mteb/fleurs", config["dataset_config"], split="test")
        if sample_size > len(dataset):
            raise ValueError(f"Requested {sample_size} samples for {language_code}, but only {len(dataset)} are available.")
        selected_indices = sorted(rng.sample(range(len(dataset)), sample_size))
        language_dir = CLEAN_AUDIO_DIR / language_code
        language_dir.mkdir(parents=True, exist_ok=True)
        for dataset_row_index in tqdm(selected_indices, desc=f"Preparing clean {config['label']}"):
            row = dataset[int(dataset_row_index)]
            utterance_id = f"{language_code}-{row['id']}"
            audio = row["audio"]
            audio_array = np.asarray(audio["array"], dtype=np.float32)
            if audio_array.ndim > 1:
                audio_array = np.mean(audio_array, axis=1)
            sampling_rate = int(audio["sampling_rate"])
            if sampling_rate != 16000:
                audio_array = librosa.resample(audio_array, orig_sr=sampling_rate, target_sr=16000)
                sampling_rate = 16000
            audio_path = language_dir / f"{utterance_id}.wav"
            sf.write(audio_path, audio_array, sampling_rate)
            manifest_rows.append(
                {
                    "utterance_id": utterance_id,
                    "language_code": language_code,
                    "language_label": config["label"],
                    "condition": "clean",
                    "dataset_name": "mteb/fleurs",
                    "dataset_config": config["dataset_config"],
                    "dataset_row_index": int(dataset_row_index),
                    "reference": row["transcription"],
                    "raw_reference": row.get("raw_transcription", row["transcription"]),
                    "gender": row.get("gender"),
                    "duration_seconds": round(len(audio_array) / sampling_rate, 6),
                    "audio_path": str(audio_path),
                    "source_audio_path": audio.get("path"),
                }
            )
    clean_manifest = pd.DataFrame(manifest_rows).sort_values(["language_code", "dataset_row_index"]).reset_index(drop=True)
    return clean_manifest


def build_noisy_manifest(clean_manifest: pd.DataFrame, seed: int = SEED) -> pd.DataFrame:
    rng = random.Random(seed)
    inventory = list_noise_inventory()
    category_pools = {
        "noise": inventory.loc[inventory["base_category"] == "noise", ["repo_id", "repo_path"]].to_dict("records"),
        "music": inventory.loc[inventory["base_category"] == "music", ["repo_id", "repo_path"]].to_dict("records"),
        "speech": inventory.loc[inventory["base_category"] == "speech", ["repo_id", "repo_path"]].to_dict("records"),
    }
    if not category_pools["noise"]:
        raise RuntimeError("Noise inventory did not contain any usable noise clips.")

    available_categories = ["noise"]
    if category_pools["music"]:
        available_categories.append("music")
    if len(category_pools["speech"]) >= 3:
        available_categories.append("babble")
    if len(available_categories) < 2:
        raise RuntimeError(
            "Noise inventory did not contain enough distinct public clips to build the requested noisy benchmark."
        )

    if "music" not in available_categories:
        display(Markdown(
            "> Pilot note: public MUSAN mirrors available in this notebook do not expose a music slice, so noisy samples are being balanced across `noise` and `babble`."
        ))

    noisy_rows = []
    for language_code, language_clean in clean_manifest.groupby("language_code", sort=True):
        language_clean = language_clean.sort_values("dataset_row_index").reset_index(drop=True)
        if len(language_clean) != SAMPLE_SIZE_PER_LANGUAGE:
            raise ValueError(f"Expected {SAMPLE_SIZE_PER_LANGUAGE} clean rows for {language_code}, found {len(language_clean)}.")

        category_assignments = (available_categories * math.ceil(len(language_clean) / len(available_categories)))[: len(language_clean)]
        snr_assignments = SNR_PLAN[: len(language_clean)]
        rng.shuffle(category_assignments)
        rng.shuffle(snr_assignments)

        pool_positions = {"noise": 0, "music": 0, "speech": 0}
        for row_idx, (_, clean_row) in enumerate(language_clean.iterrows()):
            noise_category = category_assignments[row_idx]
            snr_db = int(snr_assignments[row_idx])
            clean_audio, sampling_rate = load_mono_audio(Path(clean_row["audio_path"]), target_sr=16000)

            if noise_category == "babble":
                speech_pool = category_pools["speech"]
                noise_refs = [
                    speech_pool[(pool_positions["speech"] + offset) % len(speech_pool)]
                    for offset in range(3)
                ]
                pool_positions["speech"] += 3
                noise_signal = build_babble_signal(
                    [(item["repo_id"], item["repo_path"]) for item in noise_refs],
                    target_sr=sampling_rate,
                )
                noise_source_id = "|".join(Path(item["repo_path"]).stem for item in noise_refs)
                noise_repo_path = "|".join(item["repo_path"] for item in noise_refs)
            else:
                base_pool = category_pools[noise_category]
                noise_item = base_pool[pool_positions[noise_category] % len(base_pool)]
                pool_positions[noise_category] += 1
                noise_signal, _ = load_mono_audio(
                    ensure_noise_file(noise_item["repo_id"], noise_item["repo_path"]),
                    target_sr=sampling_rate,
                )
                noise_source_id = Path(noise_item["repo_path"]).stem
                noise_repo_path = noise_item["repo_path"]

            mixed_audio = mix_with_snr(clean_audio, noise_signal, snr_db=snr_db)
            language_dir = NOISY_AUDIO_DIR / language_code
            language_dir.mkdir(parents=True, exist_ok=True)
            noisy_path = language_dir / f"{clean_row['utterance_id']}_{noise_category}_{snr_db}db.wav"
            sf.write(noisy_path, mixed_audio, sampling_rate)

            noisy_rows.append(
                {
                    **clean_row.to_dict(),
                    "condition": "noisy",
                    "audio_path": str(noisy_path),
                    "noise_category": noise_category,
                    "noise_source_id": noise_source_id,
                    "noise_repo_path": noise_repo_path,
                    "snr_db": snr_db,
                }
            )
    noisy_manifest = pd.DataFrame(noisy_rows).sort_values(["language_code", "dataset_row_index"]).reset_index(drop=True)
    return noisy_manifest


def validate_api_access(sample_row: pd.Series):
    print(f"Running API preflight for {sample_row['language_label']} / {sample_row['condition']}...")
    result = call_stt_api(Path(sample_row["audio_path"]), sample_row["language_code"])
    if result["http_status"] in (401, 403):
        raise ValueError("Authentication failed. Update ADDIS_API_KEY and rerun the config cell.")
    if not result["success"]:
        raise ValueError(
            f"API preflight failed with status {result['http_status']}: {result['error_message']}"
        )
    print("Preflight passed.")
    return result


def run_benchmark(manifest: pd.DataFrame, label: str) -> pd.DataFrame:
    rows = manifest.to_dict(orient="records")
    results = []

    def process_row(row):
        api_result = call_stt_api(Path(row["audio_path"]), row["language_code"])
        metrics = compute_error_metrics(row["reference"], api_result["transcription"])
        latency_seconds = api_result["latency_ms"] / 1000 if pd.notna(api_result["latency_ms"]) else np.nan
        rtf = latency_seconds / row["duration_seconds"] if row["duration_seconds"] and pd.notna(latency_seconds) else np.nan
        return {
            **row,
            **api_result,
            **metrics,
            "rtf": float(rtf) if pd.notna(rtf) else np.nan,
            "usage_metadata_json": json.dumps(api_result["usage_metadata"], ensure_ascii=False) if api_result["usage_metadata"] is not None else None,
        }

    with ThreadPoolExecutor(max_workers=MAX_CONCURRENCY) as executor:
        futures = {executor.submit(process_row, row): row for row in rows}
        for future in tqdm(as_completed(futures), total=len(futures), desc=f"Benchmarking {label}"):
            results.append(future.result())

    results_df = pd.DataFrame(results).sort_values(["language_code", "condition", "dataset_row_index"]).reset_index(drop=True)
    return results_df


def build_summary_table(results_df: pd.DataFrame) -> pd.DataFrame:
    summary_rows = []
    group_columns = ["language_code", "language_label", "condition"]
    for group_keys, group in results_df.groupby(group_columns, dropna=False):
        language_code, language_label, condition = group_keys
        row = {
            "language_code": language_code,
            "language_label": language_label,
            "condition": condition,
            "requests": int(len(group)),
            "success_rate": float(group["success"].mean()),
        }
        for metric_name in [
            "wer_normalized",
            "cer_normalized",
            "wer_raw",
            "cer_raw",
            "latency_ms",
            "rtf",
            "confidence",
        ]:
            row.update(summarize_metric(group[metric_name], metric_name))
        summary_rows.append(row)
    return pd.DataFrame(summary_rows).sort_values(["language_code", "condition"]).reset_index(drop=True)


def render_headline_scorecards(summary_df: pd.DataFrame):
    headline = summary_df[[
        "language_label",
        "condition",
        "wer_normalized_mean",
        "cer_normalized_mean",
        "latency_ms_mean",
        "latency_ms_p95",
        "rtf_mean",
        "confidence_mean",
        "success_rate",
    ]].copy()
    headline = headline.rename(columns={
        "language_label": "Language",
        "condition": "Condition",
        "wer_normalized_mean": "Normalized WER",
        "cer_normalized_mean": "Normalized CER",
        "latency_ms_mean": "Mean Latency (ms)",
        "latency_ms_p95": "P95 Latency (ms)",
        "rtf_mean": "Mean RTF",
        "confidence_mean": "Mean Confidence",
        "success_rate": "Success Rate",
    })
    display(Markdown("## Headline scorecards"))
    display(headline.style.format({
        "Normalized WER": "{:.3f}",
        "Normalized CER": "{:.3f}",
        "Mean Latency (ms)": "{:.0f}",
        "P95 Latency (ms)": "{:.0f}",
        "Mean RTF": "{:.3f}",
        "Mean Confidence": "{:.3f}",
        "Success Rate": "{:.1%}",
    }))


def plot_metric_bars(summary_df: pd.DataFrame, metric: str, ci_low: str, ci_high: str, title: str, ylabel: str):
    chart_df = summary_df.copy()
    chart_df["label"] = chart_df["language_label"] + " / " + chart_df["condition"]
    order = chart_df.sort_values(["language_label", "condition"])["label"].tolist()
    positions = np.arange(len(order))
    ordered = chart_df.set_index("label").loc[order].reset_index()
    values = ordered[metric].to_numpy(dtype=float)
    lower = values - ordered[ci_low].to_numpy(dtype=float)
    upper = ordered[ci_high].to_numpy(dtype=float) - values

    plt.figure(figsize=(13, 6))
    palette = ["#0f766e" if "clean" in label else "#ea580c" for label in order]
    plt.bar(positions, values, color=palette, alpha=0.9)
    plt.errorbar(positions, values, yerr=[lower, upper], fmt="none", ecolor="black", capsize=6, lw=1.5)
    plt.xticks(positions, order, rotation=15)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.tight_layout()
    plt.show()


def render_visuals(results_df: pd.DataFrame, summary_df: pd.DataFrame):
    plot_metric_bars(
        summary_df,
        metric="wer_normalized_mean",
        ci_low="wer_normalized_ci_low",
        ci_high="wer_normalized_ci_high",
        title="Normalized WER by language and condition",
        ylabel="WER",
    )
    plot_metric_bars(
        summary_df,
        metric="cer_normalized_mean",
        ci_low="cer_normalized_ci_low",
        ci_high="cer_normalized_ci_high",
        title="Normalized CER by language and condition",
        ylabel="CER",
    )

    plt.figure(figsize=(13, 6))
    sns.violinplot(data=results_df, x="language_label", y="latency_ms", hue="condition", cut=0, inner="quartile")
    plt.title("Latency distribution")
    plt.ylabel("Latency (ms)")
    plt.xlabel("")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(13, 6))
    sns.violinplot(data=results_df, x="language_label", y="rtf", hue="condition", cut=0, inner="quartile")
    plt.title("Real-time factor distribution")
    plt.ylabel("RTF")
    plt.xlabel("")
    plt.tight_layout()
    plt.show()

    confidence_df = results_df.dropna(subset=["confidence"]).copy()
    if not confidence_df.empty:
        plt.figure(figsize=(13, 6))
        sns.violinplot(data=confidence_df, x="language_label", y="confidence", hue="condition", cut=0, inner="quartile")
        plt.title("Returned confidence distribution")
        plt.ylabel("Confidence")
        plt.xlabel("")
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(13, 6))
        sns.scatterplot(
            data=confidence_df,
            x="confidence",
            y="cer_normalized",
            hue="language_label",
            style="condition",
            s=90,
            alpha=0.8,
        )
        plt.title("Confidence vs normalized CER")
        plt.ylabel("Normalized CER")
        plt.xlabel("Confidence")
        plt.tight_layout()
        plt.show()


def render_top_failures(results_df: pd.DataFrame, top_k: int = 12):
    columns = [
        "language_label",
        "condition",
        "utterance_id",
        "reference_raw",
        "prediction_raw",
        "wer_normalized",
        "cer_normalized",
        "confidence",
        "latency_ms",
        "duration_seconds",
        "noise_category",
        "snr_db",
        "error_message",
    ]
    failures = results_df.sort_values(["wer_normalized", "cer_normalized", "latency_ms"], ascending=[False, False, False]).head(top_k)
    display(Markdown("## Top failure examples"))
    display(failures[columns].style.format({
        "wer_normalized": "{:.3f}",
        "cer_normalized": "{:.3f}",
        "confidence": "{:.3f}",
        "latency_ms": "{:.0f}",
        "duration_seconds": "{:.2f}",
    }, na_rep="-"))


In [ ]:
clean_manifest = prepare_clean_manifest(sample_size=SAMPLE_SIZE_PER_LANGUAGE, seed=SEED)
clean_manifest.to_csv(OUTPUT_DIR / "manifest_clean.csv", index=False)
display(Markdown("## Clean evaluation manifest"))
display(clean_manifest.head())
print(f"Saved clean manifest to {(OUTPUT_DIR / 'manifest_clean.csv').resolve()}")


In [ ]:
noisy_manifest = build_noisy_manifest(clean_manifest, seed=SEED)
noisy_manifest.to_csv(OUTPUT_DIR / "manifest_noisy.csv", index=False)
display(Markdown("## Noisy evaluation manifest"))
display(noisy_manifest.head())
print(f"Saved noisy manifest to {(OUTPUT_DIR / 'manifest_noisy.csv').resolve()}")


In [ ]:
preflight_result = validate_api_access(clean_manifest.iloc[0])
display(preflight_result)


In [ ]:
clean_results = run_benchmark(clean_manifest, label="clean")
clean_results.to_csv(OUTPUT_DIR / "utterance_level_results_clean.csv", index=False)
display(clean_results.head())
print(f"Saved clean utterance-level results to {(OUTPUT_DIR / 'utterance_level_results_clean.csv').resolve()}")


In [ ]:
noisy_results = run_benchmark(noisy_manifest, label="noisy")
noisy_results.to_csv(OUTPUT_DIR / "utterance_level_results_noisy.csv", index=False)
display(noisy_results.head())
print(f"Saved noisy utterance-level results to {(OUTPUT_DIR / 'utterance_level_results_noisy.csv').resolve()}")


In [ ]:
utterance_level_results = pd.concat([clean_results, noisy_results], ignore_index=True)
utterance_level_results = utterance_level_results.sort_values(["language_code", "condition", "dataset_row_index"]).reset_index(drop=True)
summary_metrics = build_summary_table(utterance_level_results)

utterance_level_results.to_csv(OUTPUT_DIR / "utterance_level_results.csv", index=False)
summary_metrics.to_csv(OUTPUT_DIR / "summary_metrics.csv", index=False)

pilot_preview_summary = {
    "generated_at_utc": pd.Timestamp.utcnow().isoformat(),
    "api_base_url": API_BASE_URL,
    "sample_size_per_language": SAMPLE_SIZE_PER_LANGUAGE,
    "seed": SEED,
    "max_concurrency": MAX_CONCURRENCY,
    "request_timeout_seconds": REQUEST_TIMEOUT_SECONDS,
    "languages": [config["label"] for config in LANGUAGE_CONFIGS.values()],
    "conditions": ["clean", "noisy"],
    "headline_metrics": summary_metrics.to_dict(orient="records"),
}

with (OUTPUT_DIR / "pilot_preview_summary.json").open("w", encoding="utf-8") as fp:
    json.dump(pilot_preview_summary, fp, ensure_ascii=False, indent=2)

render_headline_scorecards(summary_metrics)
render_visuals(utterance_level_results, summary_metrics)
render_top_failures(utterance_level_results)

print(f"Saved utterance-level results to {(OUTPUT_DIR / 'utterance_level_results.csv').resolve()}")
print(f"Saved summary metrics to {(OUTPUT_DIR / 'summary_metrics.csv').resolve()}")
print(f"Saved pilot summary JSON to {(OUTPUT_DIR / 'pilot_preview_summary.json').resolve()}")
